In [65]:
#Organize the structure of folders/ Automated Directories creation
import json
import os
from shutil import copyfile
from collections import defaultdict

# Assuming your JSON is in this format (one JSON object per line)
json_file = 'photos.json'
image_folder = 'photos/'  # Where your images are stored
output_folder = 'preprocessed_data/'

# Create output directory if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Read JSON data
data = []
with open(json_file, 'r') as f:
    for line in f:
        data.append(json.loads(line))

# Create a mapping from photo_id to label
label_map = {item['photo_id']: item['label'] for item in data}

# Count labels for information
label_counts = defaultdict(int)
print("labelcounts: ",label_counts)
for item in data:
    label_counts[item['label']] += 1

print("Label distribution:", dict(label_counts))

labelcounts:  defaultdict(<class 'int'>, {})
Label distribution: {'inside': 56031, 'outside': 18569, 'drink': 15670, 'food': 108152, 'menu': 1678}


In [67]:
import json
import os
import numpy as np
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from shutil import copyfile, rmtree
from collections import defaultdict
from tqdm import tqdm

# 1. Original Directory Organization
json_file = 'photos.json'
image_folder = 'photos/'
output_folder = 'preprocessed_data/'

os.makedirs(output_folder, exist_ok=True)

# Read JSON data
data = []
with open(json_file, 'r') as f:
    for line in f:
        data.append(json.loads(line))

# Create label mapping and counts
label_map = {item['photo_id']: item['label'] for item in data}
label_counts = defaultdict(int)

for item in data:
    label_counts[item['label']] += 1

print("Original Label Distribution:", dict(label_counts))

# 2. Create Initial Directory Structure
for label in label_counts:
    os.makedirs(os.path.join(output_folder, label), exist_ok=True)

# 3. Copy Images to Class Folders with Progress Tracking
print("\nCopying images to class folders...")
copied_files = 0
missing_files = 0

for item in tqdm(data, desc="Organizing images"):
    photo_id = item['photo_id']
    label = item['label']
    
    # Try multiple possible extensions
    source_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        temp_path = os.path.join(image_folder, photo_id + ext)
        if os.path.exists(temp_path):
            source_path = temp_path
            break
    
    if source_path:
        dest_path = os.path.join(output_folder, label, os.path.basename(source_path))
        copyfile(source_path, dest_path)
        copied_files += 1
    else:
        missing_files += 1

print(f"\nCopied {copied_files} images, {missing_files} files missing")

# 4. Class Balancing Implementation
print("\nBalancing classes...")

# Calculate target counts (25% of majority class)
majority_count = max(label_counts.values())
target_counts = {
    label: min(int(majority_count * 0.25), count) if count < majority_count * 0.25 else count
    for label, count in label_counts.items()
}

print("\nTarget Counts Per Class:")
for label, count in target_counts.items():
    print(f"{label}: {count}")

# Set up augmentation
augmentation_config = {
    'rotation_range': 20,
    'width_shift_range': 0.2,
    'height_shift_range': 0.2,
    'shear_range': 0.2,
    'zoom_range': 0.2,
    'horizontal_flip': True,
    'fill_mode': 'nearest'
}
augmenter = ImageDataGenerator(**augmentation_config)

# 5. Balance Each Class
balanced_folder = 'balanced_data'
os.makedirs(balanced_folder, exist_ok=True)

for label in label_counts:
    os.makedirs(os.path.join(balanced_folder, label), exist_ok=True)
    
    class_dir = os.path.join(output_folder, label)
    image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    current_count = len(image_files)
    target_count = target_counts[label]
    
    if current_count >= target_count:
        # Just copy files if already balanced
        for file in tqdm(image_files[:target_count], desc=f"Copying {label}"):
            src = os.path.join(class_dir, file)
            dst = os.path.join(balanced_folder, label, file)
            copyfile(src, dst)
    else:
        # Need augmentation
        copies_needed = max(1, int(target_count / current_count))
        
        print(f"\nBalancing {label}:")
        print(f"Current: {current_count}, Target: {target_count}")
        print(f"Generating {copies_needed-1} augmented copies per image")
        
        for file in tqdm(image_files, desc=label):
            src_path = os.path.join(class_dir, file)
            
            # Copy original
            copyfile(src_path, os.path.join(balanced_folder, label, file))
            
            # Generate augmented copies
            try:
                img = Image.open(src_path).convert('RGB')
                img_array = np.array(img)
                
                for i in range(1, copies_needed):
                    augmented = augmenter.random_transform(img_array)
                    aug_file = f"{os.path.splitext(file)[0]}_aug{i}.jpg"
                    Image.fromarray(augmented).save(
                        os.path.join(balanced_folder, label, aug_file),
                        quality=95
                    )
            except Exception as e:
                print(f"Error processing {file}: {str(e)}")

# 6. Verify Final Distribution
print("\nFinal Balanced Distribution:")
balanced_counts = {}
for label in label_counts:
    class_dir = os.path.join(balanced_folder, label)
    balanced_counts[label] = len([f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"{label}: {balanced_counts[label]}")

print("\nBalancing complete! Final dataset in:", balanced_folder)

Original Label Distribution: {'inside': 56031, 'outside': 18569, 'drink': 15670, 'food': 108152, 'menu': 1678}

Copying images to class folders...


Organizing images: 100%|██████████| 200100/200100 [09:02<00:00, 369.09it/s]



Copied 45146 images, 154954 files missing

Balancing classes...

Target Counts Per Class:
inside: 56031
outside: 18569
drink: 15670
food: 108152
menu: 1678

Balancing inside:
Current: 12758, Target: 56031
Generating 3 augmented copies per image


inside: 100%|██████████| 12758/12758 [34:19<00:00,  6.19it/s] 



Balancing outside:
Current: 4256, Target: 18569
Generating 3 augmented copies per image


...

Error processing bf3ymV0YgP7B6rEoriaU2w.jpg: cannot identify image file 'preprocessed_data/food\\bf3ymV0YgP7B6rEoriaU2w.jpg'


food: 100%|██████████| 24184/24184 [1:00:18<00:00,  6.68it/s]



Balancing menu:
Current: 359, Target: 1678
Generating 3 augmented copies per image


menu: 100%|██████████| 359/359 [00:41<00:00,  8.62it/s]



Final Balanced Distribution:
inside: 51032
outside: 17024
drink: 14356
food: 96661
menu: 1436

Balancing complete! Final dataset in: balanced_data


In [ ]:
#=====================>>> This point

In [71]:
import os
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
import json

# Configuration
input_json = 'photos.json'
balanced_data_dir = 'balanced_data/'  # Output from imbalance correction
output_dir = 'final_processed_data/'  # For processed images
target_size = (224, 224)

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Load JSON data
with open(input_json, 'r') as f:
    data = [json.loads(line) for line in f]

# Initialize tracking
processed_images = {}
failed_images = []

def preprocess_image(image_path, target_size=(224, 224)):
    """Enhanced preprocessing with better error handling"""
    try:
        # Try OpenCV first
        img = cv2.imread(image_path)
        if img is None:
            # Fallback to PIL if OpenCV fails
            try:
                img = np.array(Image.open(image_path).convert('RGB'))
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            except Exception as e:
                print(f"Failed to load {image_path} with PIL: {str(e)}")
                return None
        
        # Convert to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Resize with aspect ratio preservation
        h, w = img.shape[:2]
        scale = min(target_size[0]/h, target_size[1]/w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        
        # Pad if needed
        if new_h != target_size[0] or new_w != target_size[1]:
            delta_h = target_size[0] - new_h
            delta_w = target_size[1] - new_w
            top, bottom = delta_h//2, delta_h-(delta_h//2)
            left, right = delta_w//2, delta_w-(delta_w//2)
            img = cv2.copyMakeBorder(img, top, bottom, left, right, 
                                   cv2.BORDER_CONSTANT, value=[0, 0, 0])
        
        # Normalize and return
        return img.astype(np.float32) / 255.0
    
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return None

# Process all images
print(f"\nProcessing images from {balanced_data_dir}...")
for item in tqdm(data, desc="Processing Images"):
    photo_id = item['photo_id']
    label = item['label']
    
    # Find the image file (check multiple extensions)
    image_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        possible_path = os.path.join(balanced_data_dir, label, f"{photo_id}{ext}")
        if os.path.exists(possible_path):
            image_path = possible_path
            break
    
    if not image_path:
        failed_images.append(photo_id)
        continue
    
    # Preprocess the image
    processed_img = preprocess_image(image_path, target_size)
    
    if processed_img is not None:
        # Prepare output directory
        label_dir = os.path.join(output_dir, label)
        os.makedirs(label_dir, exist_ok=True)
        
        # Save processed image
        output_path = os.path.join(label_dir, f"{photo_id}.jpg")
        cv2.imwrite(
            output_path,
            cv2.cvtColor((processed_img * 255).astype(np.uint8), cv2.COLOR_RGB2BGR),
            [int(cv2.IMWRITE_JPEG_QUALITY), 95]  # Save with 95% quality
        )
        
        # Store metadata
        processed_images[photo_id] = {
            'path': output_path,
            'label': label,
            'business_id': item['business_id'],
            'caption': item['caption']
        }

# Final report
print(f"\nProcessing complete!")
print(f"Successfully processed: {len(processed_images)} images")
print(f"Failed to process: {len(failed_images)} images")

if failed_images:
    print("\nFirst 10 failed images:")
    for pid in failed_images[:10]:
        print(f"- {pid}")

# Save processing log
with open(os.path.join(output_dir, 'processing_log.json'), 'w') as f:
    json.dump({
        'processed': len(processed_images),
        'failed': len(failed_images),
        'failed_samples': failed_images,
        'class_distribution': {
            label: len([v for v in processed_images.values() if v['label'] == label])
            for label in set(item['label'] for item in data)
        }
    }, f, indent=2)

print(f"\nFinal processed data saved to: {output_dir}")


Processing images from balanced_data/...


Processing Images:   5%|▌         | 10829/200100 [00:42<08:14, 383.09it/s]

Failed to load balanced_data/food\bf3ymV0YgP7B6rEoriaU2w.jpg with PIL: cannot identify image file 'balanced_data/food\\bf3ymV0YgP7B6rEoriaU2w.jpg'


Processing Images:   7%|▋         | 13847/200100 [00:53<11:03, 280.64it/s]

Failed to load balanced_data/food\9X4YPM8nYFjf7hY8xUdc6Q.jpg with PIL: cannot identify image file 'balanced_data/food\\9X4YPM8nYFjf7hY8xUdc6Q.jpg'


...

Failed to load balanced_data/food\9BvYOtforBBP6MvvDogtmw.jpg with PIL: cannot identify image file 'balanced_data/food\\9BvYOtforBBP6MvvDogtmw.jpg'


Processing Images:  97%|█████████▋| 194257/200100 [12:05<00:19, 292.37it/s]

Failed to load balanced_data/food\ARwqGQZaT0p-XpYYjMXgQg.jpg with PIL: cannot identify image file 'balanced_data/food\\ARwqGQZaT0p-XpYYjMXgQg.jpg'
Failed to load balanced_data/food\9jBH61ndIcsheo6FtIHArA.jpg with PIL: cannot identify image file 'balanced_data/food\\9jBH61ndIcsheo6FtIHArA.jpg'


Processing Images: 100%|██████████| 200100/200100 [12:22<00:00, 269.55it/s]



Processing complete!
Successfully processed: 45121 images
Failed to process: 154954 images

First 10 failed images:
- zsvj7vloL4L5jhYyPIuVwg
- HCUdRJHHm_e0OCTlZetGLg
- vkr8T0scuJmGVvN2HJelEA
- pve7D6NUrafHW3EAORubyw
- H52Er-uBg6rNrHcReWTD2w
- wZ29mUm6nKz566j17OBadw
- QRUgAISgYLQJ9SK2yOwomw
- mcjlyGuLFJ0t4vDixycCSg
- foJzmWwl8WlC3xi-QQDRgg
- yED5k8-aiPcgiUKoPRfJgg

Final processed data saved to: final_processed_data/


In [75]:
import os
import json
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil

def create_train_val_test_splits(processed_images, output_folder='final_processed_data'):
    """
    Creates train/val/test splits from processed images and organizes them into directories
    Args:
        processed_images: Dictionary containing processed image metadata
        output_folder: Root directory where processed images are stored
    """
    
    # 1. Prepare the data for splitting
    photo_ids = list(processed_images.keys())
    labels = [processed_images[pid]['label'] for pid in photo_ids]
    
    # 2. Create stratified splits (80/10/10)
    print("\nCreating dataset splits...")
    
    # First split: 80% train, 20% temp (val+test)
    train_ids, temp_ids, train_labels, temp_labels = train_test_split(
        photo_ids, labels, test_size=0.2, random_state=42, stratify=labels)
    
    # Second split: 10% val, 10% test from the temp
    val_ids, test_ids, val_labels, test_labels = train_test_split(
        temp_ids, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)
    
    print(f"Train samples: {len(train_ids)}")
    print(f"Validation samples: {len(val_ids)}")
    print(f"Test samples: {len(test_ids)}")
    
    # 3. Create directory structure
    print("\nCreating directory structure...")
    split_dirs = ['train', 'val', 'test']
    all_labels = set(labels)
    
    for split in split_dirs:
        for label in all_labels:
            os.makedirs(os.path.join(output_folder, split, label), exist_ok=True)
    
    # 4. Enhanced file moving with progress tracking and error handling
    def organize_files(id_list, split_name):
        moved_files = 0
        missing_files = 0
        errors = 0
        
        for pid in tqdm(id_list, desc=f"Organizing {split_name} set"):
            label = processed_images[pid]['label']
            src_path = processed_images[pid]['processed_path']
            
            if not os.path.exists(src_path):
                missing_files += 1
                continue
            
            dest_path = os.path.join(
                output_folder,
                split_name,
                label,
                os.path.basename(src_path)
            )
            
            try:
                shutil.move(src_path, dest_path)
                moved_files += 1
                # Update the path in metadata
                processed_images[pid]['processed_path'] = dest_path
            except Exception as e:
                errors += 1
                print(f"Error moving {src_path}: {str(e)}")
        
        print(f"\n{split_name.capitalize()} set:")
        print(f"Successfully moved: {moved_files}")
        print(f"Missing files: {missing_files}")
        print(f"Errors: {errors}")
    
    # 5. Organize files into their respective splits
    organize_files(train_ids, 'train')
    organize_files(val_ids, 'val')
    organize_files(test_ids, 'test')
    
    # 6. Save split information for reference
    split_info = {
        'train': train_ids,
        'val': val_ids,
        'test': test_ids
    }
    
    with open(os.path.join(output_folder, 'split_info.json'), 'w') as f:
        json.dump(split_info, f, indent=2)
    
    print("\nDataset organization complete!")
    print(f"Final structure created in: {output_folder}")
    
    return processed_images

# Example usage after preprocessing:
if __name__ == "__main__":
    # Assuming you have processed_images from previous steps
    with open('photos.json', 'r') as f:
        data = [json.loads(line) for line in f]
    
    # Create processed_images dictionary (from previous steps)
    processed_images = {}
    for item in data:
        photo_id = item['photo_id']
        processed_images[photo_id] = {
            'label': item['label'],
            'business_id': item['business_id'],
            'caption': item['caption'],
            'processed_path': os.path.join('final_processed_data', item['label'], f"{photo_id}.jpg")
        }
    
    # Create the splits
    processed_images = create_train_val_test_splits(processed_images)


Creating dataset splits...
Train samples: 160078
Validation samples: 20010
Test samples: 20010

Creating directory structure...


Organizing train set: 100%|██████████| 160078/160078 [00:36<00:00, 4402.06it/s]



Train set:
Successfully moved: 36181
Missing files: 123897
Errors: 0


Organizing val set: 100%|██████████| 20010/20010 [00:04<00:00, 4354.31it/s]



Val set:
Successfully moved: 4452
Missing files: 15558
Errors: 0


Organizing test set: 100%|██████████| 20010/20010 [00:04<00:00, 4913.77it/s]



Test set:
Successfully moved: 4488
Missing files: 15522
Errors: 0

Dataset organization complete!
Final structure created in: final_processed_data


In [77]:
#Creating MetaData Files
import pandas as pd
import os
from tqdm import tqdm

def create_enhanced_metadata(processed_images, output_folder='final_processed_data'):
    """
    Creates comprehensive metadata files for train/val/test splits with additional features
    Args:
        processed_images: Dictionary containing all processed image metadata
        output_folder: Root directory of processed data
    """
    
    # 1. Load split information if available
    split_info_path = os.path.join(output_folder, 'split_info.json')
    if os.path.exists(split_info_path):
        with open(split_info_path, 'r') as f:
            split_info = json.load(f)
        train_ids = split_info['train']
        val_ids = split_info['val']
        test_ids = split_info['test']
    else:
        raise FileNotFoundError("Split information not found. Run create_train_val_test_splits() first.")
    
    # 2. Enhanced metadata creation with progress tracking
    def generate_metadata(id_list, split_name):
        metadata = []
        missing_files = 0
        
        for pid in tqdm(id_list, desc=f"Creating {split_name} metadata"):
            if pid not in processed_images:
                missing_files += 1
                continue
                
            img_data = processed_images[pid]
            
            # Check if file actually exists
            file_path = os.path.join(output_folder, split_name, img_data['label'], f"{pid}.jpg")
            if not os.path.exists(file_path):
                # Check for alternate extensions
                found = False
                for ext in ['.jpg', '.jpeg', '.png']:
                    alt_path = os.path.join(output_folder, split_name, img_data['label'], f"{pid}{ext}")
                    if os.path.exists(alt_path):
                        file_path = alt_path
                        found = True
                        break
                
                if not found:
                    missing_files += 1
                    continue
            
            # Create relative path for portability
            rel_path = os.path.relpath(file_path, output_folder)
            
            metadata.append({
                'photo_id': pid,
                'path': rel_path.replace("\\", "/"),  # Standardize path separators
                'absolute_path': os.path.abspath(file_path),
                'label': img_data['label'],
                'business_id': img_data['business_id'],
                'caption': img_data['caption'],
                'split': split_name,
                'file_size': os.path.getsize(file_path),
                'modified_time': os.path.getmtime(file_path)
            })
        
        if missing_files > 0:
            print(f"Warning: {missing_files} files missing from {split_name} set")
        
        return pd.DataFrame(metadata)
    
    # 3. Generate metadata for each split
    print("\nCreating metadata files...")
    train_meta = generate_metadata(train_ids, 'train')
    val_meta = generate_metadata(val_ids, 'val')
    test_meta = generate_metadata(test_ids, 'test')
    
    # 4. Save with additional options
    meta_dir = os.path.join(output_folder, 'metadata')
    os.makedirs(meta_dir, exist_ok=True)
    
    train_meta.to_csv(
        os.path.join(meta_dir, 'train_metadata.csv'),
        index=False,
        encoding='utf-8'
    )
    val_meta.to_csv(
        os.path.join(meta_dir, 'val_metadata.csv'),
        index=False,
        encoding='utf-8'
    )
    test_meta.to_csv(
        os.path.join(meta_dir, 'test_metadata.csv'),
        index=False,
        encoding='utf-8'
    )
    
    # 5. Create combined metadata file
    combined_meta = pd.concat([train_meta, val_meta, test_meta])
    combined_meta.to_csv(
        os.path.join(meta_dir, 'combined_metadata.csv'),
        index=False,
        encoding='utf-8'
    )
    
    # 6. Save statistics
    stats = {
        'total_samples': len(combined_meta),
        'train_samples': len(train_meta),
        'val_samples': len(val_meta),
        'test_samples': len(test_meta),
        'class_distribution': combined_meta['label'].value_counts().to_dict()
    }
    
    with open(os.path.join(meta_dir, 'dataset_stats.json'), 'w') as f:
        json.dump(stats, f, indent=2)
    
    print("\nMetadata creation complete!")
    print(f"Files saved in: {meta_dir}")
    
    return {
        'train': train_meta,
        'val': val_meta,
        'test': test_meta,
        'combined': combined_meta
    }

# Example usage:
if __name__ == "__main__":
    # Assuming processed_images is available from previous steps
    metadata = create_enhanced_metadata(processed_images)


Creating metadata files...


Creating train metadata: 100%|██████████| 160078/160078 [00:38<00:00, 4154.87it/s]


Creating val metadata: 100%|██████████| 20010/20010 [00:04<00:00, 4465.84it/s]


Creating test metadata: 100%|██████████| 20010/20010 [00:04<00:00, 4502.11it/s]



Metadata creation complete!
Files saved in: final_processed_data\metadata
